# Build a deliberately skewed join, watch Spark UI surface the bottleneck, then enable AQE skew-join handling and see the stage rebalance

A junior engineer's join is taking 40 minutes on a 1 GB dataset and the senior wants the team to know how to identify whether the cause is data skew, an over-aggressive shuffle, or disk spilling before reaching for "increase cluster size." You have one session to build a deliberately skewed join, look at the Spark UI like an actual senior engineer would, find the outlier task, then flip the AQE flag and watch the stage rebalance. After this lab, "increase cluster size" is no longer your first move.

The hands-on work:

- Create a Unity Catalog schema (`workspace.default.labrun_skew`) plus a small dimension table (`dim_users`, 100 rows) and a deliberately skewed fact table (`fact_events`, 50,000 rows where 90 percent carry `user_id=1`).
- Run the same inner join twice on the Free Edition serverless SQL warehouse: first with `spark.sql.adaptive.skewJoin.enabled=false`, then with it `=true` plus the lower per-partition byte threshold (`10MB`) so the skew handler actually fires on this small dataset.
- Capture per-run metrics (statement id, elapsed time, task counts derived from query history) into a `run_metrics` table so the comparison is rigorous, not vibes.
- Write a one-paragraph diagnostic observation that names skew vs shuffle vs spill so the same junior engineer sees how a senior reasons about a slow stage.

**Time.** Plan for about 55 minutes hands-on. Most of that is the second join run plus reading the query history JSON. Budget up to 90 minutes for the session window.

**Cost.** Zero dollars. Two skewed-join runs on a 50K-row fact table are not how you blow a Free Edition daily quota. The point is the muscle memory of opening Spark UI before reaching for "bigger cluster," not the dollar spend.

This lab maps to Databricks DE Associate Domain 3 (Transformation, ~31 percent provisional) and Domain 6 (Troubleshooting, Monitoring, and Optimization, ~8 percent provisional).

**Rename callout.** "Adaptive Query Execution" is current; the AQE flag namespace has not been renamed. If your other prep material says "Liquid Clustering replaces AQE skew handling," that is wrong: Liquid Clustering is a TABLE-level layout strategy and AQE skew handling is a QUERY-level runtime optimization. The reflection MCQ at the bottom tests this distinction.

**Local prerequisite.** This lab runs from Colab against your Databricks Free Edition workspace via the Statement Execution API. You will paste two getpass prompts: your workspace URL (something like `https://dbc-xxxxxxxx.cloud.databricks.com`) and a personal access token from the workspace User Settings page.

In [ ]:
# NBVAL_SKIP
# Install labrun-checks and the Databricks SDK. Pinned versions per
# LAB-CREATION-BLUEPRINT.md build rules.

!pip install --quiet labrun-checks==0.6.0 databricks-sdk==0.40.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. The lab runs SQL through the Free Edition
# Starter SQL warehouse via the Statement Execution API, then pulls per-query
# metrics via the SQL Query History API.

import atexit
import getpass
import json
import statistics
import sys
import time

import requests

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import StatementState

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupResource,
)

LAB_ID = "spark-ui-skew-and-aqe"
LAB_TAG_KEY = "labrun_lab_slug"
LAB_TAG_VALUE = LAB_ID

CATALOG = "workspace"
PARENT_SCHEMA = "default"
LAB_SCHEMA = "labrun_skew"
SCHEMA_FQN = f"{CATALOG}.{PARENT_SCHEMA}.{LAB_SCHEMA}"

DIM_TABLE_FQN = f"{SCHEMA_FQN}.dim_users"
FACT_TABLE_FQN = f"{SCHEMA_FQN}.fact_events"
METRICS_TABLE_FQN = f"{SCHEMA_FQN}.run_metrics"

# Skew shape: 50K rows total, 45K (90%) on user_id=1.
FACT_ROW_COUNT = 50000
SKEW_KEY = 1
SKEW_ROW_COUNT = 45000

# Set by the student during Tasks 2 and 3.
baseline_run_metrics = None
aqe_run_metrics = None
skew_observation_text = None

In [ ]:
# NBVAL_SKIP
# Register the session, validate Databricks credentials, resolve the
# Starter SQL warehouse, expose run_sql() and a query-history fetcher.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
databricks_host = getpass.getpass("Databricks workspace URL: ").strip()
databricks_token = getpass.getpass("Databricks personal access token (PAT): ").strip()

if not databricks_host or not databricks_token:
    print("Workspace URL and PAT are both required. Re-run this cell.")
    raise SystemExit(1)
if not databricks_host.startswith("https://"):
    databricks_host = f"https://{databricks_host}"
databricks_host = databricks_host.rstrip("/")

w = WorkspaceClient(host=databricks_host, token=databricks_token)

try:
    me = w.current_user.me()
except Exception as exc:
    print("Databricks credentials are missing or expired:")
    print(f"  {exc}")
    raise SystemExit(1)
CALLER_USER = me.user_name or (me.emails[0].value if me.emails else "unknown")
print(f"Credentials validated. Workspace user: {CALLER_USER}")

warehouses = list(w.warehouses.list())
if not warehouses:
    print("No SQL warehouses found. Recreate the Starter warehouse and re-run.")
    raise SystemExit(1)
WAREHOUSE = warehouses[0]
WAREHOUSE_ID = WAREHOUSE.id
print(f"SQL warehouse resolved: {WAREHOUSE.name} ({WAREHOUSE_ID})")

DATABRICKS_CREDENTIALS = {
    "host": databricks_host,
    "token": databricks_token,
    "warehouse_id": WAREHOUSE_ID,
}


def run_sql(statement, warehouse_id=None, wait_seconds=240, return_statement_id=False):
    wid = warehouse_id or WAREHOUSE_ID
    resp = w.statement_execution.execute_statement(
        warehouse_id=wid, statement=statement, wait_timeout="50s",
    )
    statement_id = resp.statement_id
    deadline = time.time() + wait_seconds
    while True:
        state = resp.status.state if resp.status else None
        if state == StatementState.SUCCEEDED:
            break
        if state in (StatementState.FAILED, StatementState.CANCELED, StatementState.CLOSED):
            err = resp.status.error.message if (resp.status and resp.status.error) else "no error message"
            raise RuntimeError(f"SQL failed ({state}): {err}\n  Statement: {statement[:200]}")
        if time.time() > deadline:
            raise TimeoutError(f"SQL did not finish in {wait_seconds}s. Last state: {state}.")
        time.sleep(2)
        resp = w.statement_execution.get_statement(statement_id)
    columns = []
    if resp.manifest and resp.manifest.schema and resp.manifest.schema.columns:
        columns = [c.name for c in resp.manifest.schema.columns]
    rows = []
    if resp.result and resp.result.data_array:
        rows = list(resp.result.data_array)
    out = {"columns": columns, "rows": rows, "statement_id": statement_id}
    return out


def get_query_history(statement_id, retries=6, delay_seconds=5):
    """Fetch query-history record for a recent statement. The history API
    is eventually consistent on the order of seconds; retry briefly."""
    url = f"{databricks_host}/api/2.0/sql/history/queries"
    headers = {"Authorization": f"Bearer {databricks_token}"}
    params = {
        "filter_by": json.dumps({"statement_ids": [statement_id]}),
        "include_metrics": "true",
    }
    for attempt in range(retries):
        try:
            resp = requests.get(url, headers=headers, params=params, timeout=30)
            resp.raise_for_status()
            queries = resp.json().get("res", [])
            if queries:
                return queries[0]
        except Exception:
            pass
        time.sleep(delay_seconds)
    return None


register(session_token=session_token, lab_id=LAB_ID, credentials=DATABRICKS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan. No critical
# resources in this lab: Free Edition serverless auto-pauses after each
# cell completes, so nothing bills hourly. Cleanup just drops the three
# tables and the schema in reverse-creation order.


CLEANUP_MANIFEST = [
    CleanupResource(
        type="uc_managed_table",
        id=METRICS_TABLE_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {METRICS_TABLE_FQN}\"",
    ),
    CleanupResource(
        type="uc_managed_table",
        id=FACT_TABLE_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {FACT_TABLE_FQN}\"",
    ),
    CleanupResource(
        type="uc_managed_table",
        id=DIM_TABLE_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {DIM_TABLE_FQN}\"",
    ),
    CleanupResource(
        type="uc_schema",
        id=SCHEMA_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP SCHEMA IF EXISTS {SCHEMA_FQN} CASCADE\"",
    ),
]


class DatabricksCleanupAdapter:
    def delete_resource(self, credentials, resource):
        rtype, rid = resource.type, resource.id
        if rtype == "uc_managed_table":
            run_sql(f"DROP TABLE IF EXISTS {rid}")
        elif rtype == "uc_schema":
            run_sql(f"DROP SCHEMA IF EXISTS {rid} CASCADE")
        else:
            raise ValueError(f"Unknown resource type {rtype!r}")

    def describe_resource(self, credentials, resource):
        rtype, rid = resource.type, resource.id
        if rtype == "uc_managed_table":
            catalog, schema, table = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.tables "
                f"WHERE table_catalog = '{catalog}' AND table_schema = '{schema}' "
                f"AND table_name = '{table}'"
            )
            return len(run_sql(sql)["rows"]) > 0
        if rtype == "uc_schema":
            catalog, schema = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.schemata "
                f"WHERE catalog_name = '{catalog}' AND schema_name = '{schema}'"
            )
            return len(run_sql(sql)["rows"]) > 0
        return False

    def tag_scan(self, credentials, lab_slug, region):
        arns = []
        queries = [
            (
                "SELECT catalog_name, schema_name FROM system.information_schema.schema_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}",
            ),
            (
                "SELECT catalog_name, schema_name, table_name FROM system.information_schema.table_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}.{row[2]}",
            ),
        ]
        for sql, fmt in queries:
            try:
                result = run_sql(sql)
            except Exception:
                continue
            for row in result["rows"]:
                arns.append(fmt(row))
        return arns


CLEANUP_ADAPTER = DatabricksCleanupAdapter()


def _atexit_cleanup():
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


orphans = CLEANUP_ADAPTER.tag_scan(DATABRICKS_CREDENTIALS, LAB_TAG_VALUE, "databricks")
if orphans:
    print(f"BLOCKED: existing objects tagged {LAB_TAG_KEY}={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("Run the cleanup cell at the bottom of this notebook first.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Task 1: Build the schema, the dimension table, and the deliberately skewed fact table

The lab starts with three SQL objects. The shape of `fact_events` is the whole pedagogical point: 50,000 rows total, 45,000 of them carry `user_id = 1`. That's the deliberate skew. The other 5,000 rows spread evenly across `user_id` values 2 through 100.

Build it as one CREATE TABLE plus two INSERTs (or one INSERT with a UNION). The lab gives you the `range()` and `sequence()` patterns you need; the join behavior in Tasks 2 and 3 only works if the skew shape is what we said it is.

Tag the schema and all three tables with `labrun_lab_slug = spark-ui-skew-and-aqe` so the orphan scan finds them on a future re-run.

Also pre-create the empty `run_metrics` table so Tasks 2 and 3 can INSERT into it without an extra DDL step mid-task.

In [ ]:
# NBVAL_SKIP
# Task 1: Create the schema, dim_users (100 distinct ids), fact_events
# (50K rows with 90% on user_id=1), and the empty run_metrics table.
# Tag everything with the lab slug.

# YOUR CODE: CREATE SCHEMA IF NOT EXISTS SCHEMA_FQN via run_sql().
# YOUR CODE: ALTER SCHEMA SCHEMA_FQN SET TAGS ('LAB_TAG_KEY' = 'LAB_TAG_VALUE').

# YOUR CODE: CREATE OR REPLACE TABLE DIM_TABLE_FQN AS
#            SELECT id AS user_id, concat('user_', id) AS user_name
#            FROM range(1, 101)
# (range(start, end) is half-open: 1..100 inclusive yields 100 rows.)
# YOUR CODE: ALTER TABLE DIM_TABLE_FQN SET TAGS.

# Build fact_events as the union of the skewed slice (user_id=1, 45K rows)
# and the spread slice (user_id 2..100, 5K rows divided ~evenly).
# The numeric expressions matter; they create the deliberate skew.
# YOUR CODE: CREATE OR REPLACE TABLE FACT_TABLE_FQN AS
#   SELECT CAST(SKEW_KEY AS INT) AS user_id,
#          concat('event_', id) AS event_id,
#          CAST(id AS BIGINT) AS event_value
#   FROM range(1, SKEW_ROW_COUNT + 1)
#   UNION ALL
#   SELECT CAST(((id - 1) % 99) + 2 AS INT) AS user_id,
#          concat('event_other_', id) AS event_id,
#          CAST(id AS BIGINT) AS event_value
#   FROM range(1, FACT_ROW_COUNT - SKEW_ROW_COUNT + 1)
# YOUR CODE: ALTER TABLE FACT_TABLE_FQN SET TAGS.

# YOUR CODE: CREATE TABLE IF NOT EXISTS METRICS_TABLE_FQN (
#   aqe_enabled BOOLEAN,
#   statement_id STRING,
#   elapsed_time_ms BIGINT,
#   task_count INT,
#   max_task_shuffle_bytes BIGINT,
#   median_task_shuffle_bytes BIGINT,
#   captured_at TIMESTAMP
# ) USING DELTA
# YOUR CODE: ALTER TABLE METRICS_TABLE_FQN SET TAGS.

print(f"Schema:        {SCHEMA_FQN}")
print(f"Dim table:     {DIM_TABLE_FQN}")
print(f"Fact table:    {FACT_TABLE_FQN}")
print(f"Metrics table: {METRICS_TABLE_FQN}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: Schema and tables exist and are tagged. dim_users has 100
# distinct user_ids in [1, 100]. fact_events has 50,000 rows with at least
# 44,000 on user_id=1 (90% target with tolerance). run_metrics exists and
# is empty.


def checkpoint_1(session):
    try:
        catalog, parent, schema = SCHEMA_FQN.split(".")
        schema_sql = (
            "SELECT 1 FROM system.information_schema.schemata "
            f"WHERE catalog_name = '{catalog}' AND schema_name = '{schema}'"
        )
        if not run_sql(schema_sql)["rows"]:
            return CheckpointResult(
                status="fail",
                error_reason=f"Schema {SCHEMA_FQN} not found. Did CREATE SCHEMA run?",
            )

        for table_fqn in (DIM_TABLE_FQN, FACT_TABLE_FQN, METRICS_TABLE_FQN):
            cat, par, tbl = table_fqn.split(".")
            tbl_sql = (
                "SELECT 1 FROM system.information_schema.tables "
                f"WHERE table_catalog = '{cat}' AND table_schema = '{par}' "
                f"AND table_name = '{tbl}'"
            )
            if not run_sql(tbl_sql)["rows"]:
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Table {table_fqn} not found.",
                )

        # Tag presence on schema.
        tag_sql = (
            "SELECT tag_value FROM system.information_schema.schema_tags "
            f"WHERE catalog_name = '{catalog}' AND schema_name = '{schema}' "
            f"AND tag_name = '{LAB_TAG_KEY}'"
        )
        if not any(r[0] == LAB_TAG_VALUE for r in run_sql(tag_sql)["rows"]):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Schema {SCHEMA_FQN} missing tag "
                    f"{LAB_TAG_KEY}={LAB_TAG_VALUE}. The orphan scan needs it."
                ),
            )

        # dim_users shape: 100 rows with distinct user_id in [1, 100].
        dim_count = run_sql(f"SELECT COUNT(*), COUNT(DISTINCT user_id), MIN(user_id), MAX(user_id) FROM {DIM_TABLE_FQN}")["rows"][0]
        total, distinct, mn, mx = dim_count
        if int(total) != 100:
            return CheckpointResult(
                status="fail",
                error_reason=f"dim_users has {total} rows; expected exactly 100.",
            )
        if int(distinct) != 100:
            return CheckpointResult(
                status="fail",
                error_reason=f"dim_users has {distinct} distinct user_id values; expected 100.",
            )
        if int(mn) != 1 or int(mx) != 100:
            return CheckpointResult(
                status="fail",
                error_reason=f"dim_users user_id range is [{mn}, {mx}]; expected [1, 100].",
            )

        # fact_events shape: 50K total, >=44K on user_id=1 (90% target).
        fact_count = run_sql(f"SELECT COUNT(*) FROM {FACT_TABLE_FQN}")["rows"][0][0]
        if int(fact_count) != FACT_ROW_COUNT:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"fact_events has {fact_count} rows; expected exactly "
                    f"{FACT_ROW_COUNT}."
                ),
            )
        skew_count = run_sql(
            f"SELECT COUNT(*) FROM {FACT_TABLE_FQN} WHERE user_id = {SKEW_KEY}"
        )["rows"][0][0]
        if int(skew_count) < 44000:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"fact_events has only {skew_count} rows with user_id={SKEW_KEY}; "
                    f"need at least 44000 to get the 9x skew the lab demonstrates."
                ),
            )

        # run_metrics must be empty so Tasks 2 and 3 INSERT cleanly.
        metrics_count = run_sql(f"SELECT COUNT(*) FROM {METRICS_TABLE_FQN}")["rows"][0][0]
        if int(metrics_count) > 0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"run_metrics already has {metrics_count} rows. Drop and recreate "
                    f"the table so Tasks 2 and 3 INSERT a clean baseline."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint names what is missing: a schema row, a tag, a table, the dim row count, the fact row count, the skewed-key row count, or a non-empty run_metrics. Each is a separate SQL.

</details>

<details><summary>Hint 2 (direction)</summary>

Use `range(1, 101)` for a sequence-of-integers source on the SQL warehouse. To build a skewed fact table, write the 45,000 skewed rows in one SELECT and the 5,000 spread rows in a second SELECT, then UNION ALL them. The skewed slice carries `CAST(1 AS INT) AS user_id`; the spread slice computes `((id - 1) % 99) + 2` so values land in `[2, 100]`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
run_sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_FQN}")
run_sql(f"ALTER SCHEMA {SCHEMA_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")

run_sql(
    f"CREATE OR REPLACE TABLE {DIM_TABLE_FQN} AS "
    f"SELECT CAST(id AS INT) AS user_id, concat('user_', id) AS user_name "
    f"FROM range(1, 101)"
)
run_sql(f"ALTER TABLE {DIM_TABLE_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")

run_sql(
    f"CREATE OR REPLACE TABLE {FACT_TABLE_FQN} AS "
    f"SELECT CAST({SKEW_KEY} AS INT) AS user_id, "
    f"       concat('event_', id) AS event_id, "
    f"       CAST(id AS BIGINT) AS event_value "
    f"FROM range(1, {SKEW_ROW_COUNT + 1}) "
    f"UNION ALL "
    f"SELECT CAST(((id - 1) % 99) + 2 AS INT) AS user_id, "
    f"       concat('event_other_', id) AS event_id, "
    f"       CAST(id AS BIGINT) AS event_value "
    f"FROM range(1, {FACT_ROW_COUNT - SKEW_ROW_COUNT + 1})"
)
run_sql(f"ALTER TABLE {FACT_TABLE_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")

run_sql(
    f"CREATE TABLE IF NOT EXISTS {METRICS_TABLE_FQN} ("
    f"  aqe_enabled BOOLEAN, statement_id STRING, elapsed_time_ms BIGINT, "
    f"  task_count INT, max_task_shuffle_bytes BIGINT, "
    f"  median_task_shuffle_bytes BIGINT, captured_at TIMESTAMP"
    f") USING DELTA"
)
run_sql(f"ALTER TABLE {METRICS_TABLE_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")
```

</details>

**Wallet check.** Still at $0.00. Three small Delta tables and one empty metrics table sit comfortably inside the Free Edition daily compute quota. Two CREATEs and a few INSERT-equivalents on a serverless warehouse cost a fraction of a DBU each. Your morning coffee cost more than the entire setup.

## Task 2: Run the join WITHOUT AQE skew handling, then read the per-stage metrics

This is the code half of a `code_observe` task block. You will:

1. Disable AQE skew-join handling explicitly (`SET spark.sql.adaptive.skewJoin.enabled = false`). Recent Databricks runtimes ship `spark.sql.adaptive.enabled = true` by default; this lab turns OFF the skew piece specifically so the comparison in Task 3 is clean.
2. Run the inner join `fact_events JOIN dim_users ON user_id` and capture the elapsed time. Use `SELECT COUNT(*) FROM ... JOIN ...` so the warehouse actually shuffles all rows, not just the first matching one.
3. Capture the `statement_id` returned by `run_sql()`. The next cell (the OBSERVE half) uses it to fetch per-task metrics from the SQL Query History API.

The OBSERVE cell pulls metrics from `GET /api/2.0/sql/history/queries?include_metrics=true&filter_by={"statement_ids":[...]}`. The full per-task shuffle distribution is in the workspace's Spark UI for the warehouse query (open the query in the SQL editor history page if you want to see it visually); the lab uses the API path so the validator stays workspace-agnostic.

You then INSERT one row into `run_metrics` capturing the baseline numbers. Task 3 INSERTs the AQE row.

In [ ]:
# NBVAL_SKIP
# Task 2 code phase: SET the AQE skew-join flag false, run the join, capture
# the statement_id and elapsed time. The next cell (observe phase) pulls
# per-task metrics for this statement_id and INSERTs into run_metrics.

global baseline_run_metrics

# YOUR CODE: Run `SET spark.sql.adaptive.skewJoin.enabled = false` via run_sql().

# YOUR CODE: Build the join SQL as a multi-line string. Use COUNT(*) on the
# JOIN so the warehouse shuffles all rows.
#   SELECT COUNT(*) FROM fact_events f INNER JOIN dim_users d
#   ON f.user_id = d.user_id
join_sql = None  # YOUR CODE

# YOUR CODE: Capture wall-clock time around run_sql(join_sql).
# baseline_started = time.time()
# baseline_result = run_sql(join_sql, wait_seconds=300)
# baseline_elapsed_ms = int((time.time() - baseline_started) * 1000)

# YOUR CODE: Assign baseline_run_metrics = {
#   "statement_id": baseline_result["statement_id"],
#   "elapsed_ms": baseline_elapsed_ms,
#   "row_count": int(baseline_result["rows"][0][0]) if baseline_result["rows"] else 0,
# }

if baseline_run_metrics is None:
    print("baseline_run_metrics is None. Run the SET, the join, and assign the dict.")
else:
    print(f"Baseline join statement_id: {baseline_run_metrics['statement_id']}")
    print(f"Baseline elapsed:           {baseline_run_metrics['elapsed_ms']} ms")
    print(f"Baseline row count:         {baseline_run_metrics['row_count']}")

In [ ]:
# NBVAL_SKIP
# Task 2 observe phase: Pull per-task metrics from the SQL Query History
# API for the baseline statement_id. Print the metrics breakdown so you
# see the skew shape before INSERTing the row into run_metrics.

if baseline_run_metrics is None:
    raise RuntimeError("Run the previous cell (Task 2 code phase) first.")

baseline_history = get_query_history(baseline_run_metrics["statement_id"])
if baseline_history is None:
    raise RuntimeError(
        "Query history did not surface metrics for the baseline statement. "
        "Wait 30 seconds and re-run this cell; the history API is eventually "
        "consistent."
    )

baseline_metrics = baseline_history.get("metrics") or {}
# The history API does not expose per-task shuffle bytes directly; use
# the read partitions count plus the per-partition byte estimate as a
# pragmatic proxy. The skewed partition holds 45K of 50K rows, so the
# max-vs-median ratio reflects the row distribution.
baseline_partitions = int(baseline_metrics.get("read_partitions", 0) or 0)
baseline_read_bytes = int(baseline_metrics.get("read_bytes", 0) or 0)
baseline_total_time_ms = int(baseline_metrics.get("total_time_ms", 0) or 0)

# Per-key row counts let us reconstruct the per-partition shuffle proxy.
per_key = run_sql(
    f"SELECT user_id, COUNT(*) FROM {FACT_TABLE_FQN} GROUP BY user_id ORDER BY user_id"
)
key_counts = [int(r[1]) for r in per_key["rows"]]
max_count = max(key_counts) if key_counts else 0
median_count = int(statistics.median(key_counts)) if key_counts else 0
# Approximate per-task shuffle bytes by row count weighted by avg row size.
avg_row_bytes = (baseline_read_bytes // sum(key_counts)) if sum(key_counts) else 80
max_task_shuffle_bytes = max_count * max(avg_row_bytes, 1)
median_task_shuffle_bytes = median_count * max(avg_row_bytes, 1)
inferred_task_count = baseline_partitions or len(key_counts)

# YOUR CODE: INSERT into run_metrics with aqe_enabled=false plus the
# captured numbers. Use run_sql() with a parameterized INSERT.
# Example:
#   run_sql(f"INSERT INTO {METRICS_TABLE_FQN} VALUES (false, '{statement_id}', "
#           f"{elapsed_ms}, {task_count}, {max_bytes}, {median_bytes}, current_timestamp())")

print(f"Baseline statement_id: {baseline_run_metrics['statement_id']}")
print(f"Baseline elapsed_ms (wall clock):   {baseline_run_metrics['elapsed_ms']}")
print(f"Baseline total_time_ms (warehouse): {baseline_total_time_ms}")
print(f"Baseline partitions read:           {baseline_partitions}")
print(f"Inferred task count:                {inferred_task_count}")
print(f"Max-key row count (skewed key):     {max_count}")
print(f"Median-key row count:               {median_count}")
print(f"Max/median ratio:                   {(max_count / max(median_count, 1)):.1f}x")
print(f"Max task shuffle bytes (estimate):  {max_task_shuffle_bytes}")
print(f"Median task shuffle bytes (est.):   {median_task_shuffle_bytes}")
print()
print("Open the same query in your workspace SQL editor history view to see")
print("the live Spark UI stage page. The outlier task on the skewed key is")
print("what you would point at in a real perf review.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: run_metrics has exactly one row with aqe_enabled=false.
# The captured numbers prove (a) the join ran, (b) the skew shape shows
# at least a 5x ratio between max and median per-key shuffle bytes,
# (c) the elapsed time is non-trivial.


def checkpoint_2(session):
    try:
        rows = run_sql(
            f"SELECT statement_id, elapsed_time_ms, task_count, "
            f"max_task_shuffle_bytes, median_task_shuffle_bytes "
            f"FROM {METRICS_TABLE_FQN} WHERE aqe_enabled = false"
        )["rows"]
        if not rows:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "run_metrics has no row with aqe_enabled=false. INSERT the "
                    "baseline metrics from Task 2 observe phase before checking."
                ),
            )
        if len(rows) > 1:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"run_metrics has {len(rows)} rows with aqe_enabled=false; "
                    f"expected exactly 1. Drop the table and re-run Task 2."
                ),
            )
        statement_id, elapsed_ms, task_count, max_bytes, median_bytes = rows[0]
        if not statement_id or not isinstance(statement_id, str):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "Baseline statement_id is missing. Capture it from "
                    "run_sql()'s return value."
                ),
            )
        if int(elapsed_ms or 0) <= 0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Baseline elapsed_time_ms is {elapsed_ms}; capture wall-clock "
                    f"time around the join."
                ),
            )
        max_b = int(max_bytes or 0)
        med_b = int(median_bytes or 1)
        if med_b <= 0:
            return CheckpointResult(
                status="fail",
                error_reason="median_task_shuffle_bytes must be > 0.",
            )
        ratio = max_b / med_b
        if ratio < 5.0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"max/median shuffle-bytes ratio is {ratio:.1f}; expected >=5x. "
                    f"Confirm fact_events still has 45000 rows on user_id=1; the "
                    f"skew shape is what produces the outlier."
                ),
            )
        # Confirm the statement actually exists in query history (proves the
        # student did not fabricate the row).
        history = get_query_history(statement_id, retries=2, delay_seconds=2)
        if history is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"statement_id {statement_id!r} was not found in SQL Query "
                    f"History. Did you paste a fake id?"
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint names exactly what is missing: the SET, the join, the wall-clock capture, the INSERT, or the skew ratio. Each is one SQL or one Python line.

</details>

<details><summary>Hint 2 (direction)</summary>

`run_sql()` returns a dict including `statement_id`. Wrap your join call in `time.time()` calls to capture wall-clock elapsed. Then INSERT one row into `run_metrics` with `aqe_enabled=false` plus the four numbers from the observe cell (`elapsed_ms`, `inferred_task_count`, `max_task_shuffle_bytes`, `median_task_shuffle_bytes`). Use `current_timestamp()` for the captured_at column.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
global baseline_run_metrics

run_sql("SET spark.sql.adaptive.skewJoin.enabled = false")

join_sql = (
    f"SELECT COUNT(*) FROM {FACT_TABLE_FQN} f "
    f"INNER JOIN {DIM_TABLE_FQN} d ON f.user_id = d.user_id"
)
baseline_started = time.time()
baseline_result = run_sql(join_sql, wait_seconds=300)
baseline_elapsed_ms = int((time.time() - baseline_started) * 1000)
baseline_run_metrics = {
    "statement_id": baseline_result["statement_id"],
    "elapsed_ms": baseline_elapsed_ms,
    "row_count": int(baseline_result["rows"][0][0]) if baseline_result["rows"] else 0,
}
```

Then in the observe cell, after the metric prints, INSERT:

```python
run_sql(
    f"INSERT INTO {METRICS_TABLE_FQN} VALUES ("
    f"  false, '{baseline_run_metrics['statement_id']}', "
    f"  {baseline_run_metrics['elapsed_ms']}, {inferred_task_count}, "
    f"  {max_task_shuffle_bytes}, {median_task_shuffle_bytes}, "
    f"  current_timestamp()"
    f")"
)
```

</details>

**Wallet check.** Still at $0.00. One join on a 50K-row fact table costs a fraction of a DBU on Free Edition serverless. The query history API call is free. Cumulative session damage: less than your morning coffee 1000x over.

## Task 3: Re-run the same join WITH AQE skew handling and confirm the partition split

Now flip the flag on and re-run the same join. Three SQL settings together make AQE actually fire on this small dataset:

- `spark.sql.adaptive.skewJoin.enabled = true` (the master switch)
- `spark.sql.adaptive.skewJoin.skewedPartitionFactor = 5` (a partition is "skewed" if it is 5x the median; the default; left explicit so the lab is self-contained)
- `spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes = 10MB` (the bytes ceiling above which a partition is eligible for splitting; default 256MB, lowered for the lab so the small-data join actually triggers the handler)

Run the same `SELECT COUNT(*) FROM fact_events JOIN dim_users` and capture metrics the same way. The OBSERVE cell pulls history for the new statement_id and INSERTs the AQE row.

You will see one of two results:

- The AQE row's `task_count` is greater than the baseline's (the skewed partition was split into subtasks). This is the textbook win.
- The AQE row's `elapsed_time_ms` is comparable or slightly lower than the baseline. On a 50K-row dataset the overhead of skew handling can wash the win, so the lab is loose on the time comparison; the task-count delta is the real evidence.

In [ ]:
# NBVAL_SKIP
# Task 3 code phase: enable AQE skew-join handling with the lower per-partition
# byte threshold so the handler actually fires on this small dataset, then
# re-run the same join and capture metrics.

global aqe_run_metrics

# YOUR CODE: Three SET statements for the AQE flags above.
#   run_sql("SET spark.sql.adaptive.skewJoin.enabled = true")
#   run_sql("SET spark.sql.adaptive.skewJoin.skewedPartitionFactor = 5")
#   run_sql("SET spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes = 10MB")

# YOUR CODE: Re-use the same join SQL from Task 2.
join_sql_aqe = None  # YOUR CODE

# YOUR CODE: Wall-clock-time the run_sql() call. Assign aqe_run_metrics dict
# with the same keys as baseline_run_metrics.

if aqe_run_metrics is None:
    print("aqe_run_metrics is None. Run the three SETs, the join, and assign the dict.")
else:
    print(f"AQE join statement_id: {aqe_run_metrics['statement_id']}")
    print(f"AQE elapsed:           {aqe_run_metrics['elapsed_ms']} ms")
    print(f"AQE row count:         {aqe_run_metrics['row_count']}")

In [ ]:
# NBVAL_SKIP
# Task 3 observe phase: Pull per-task metrics for the AQE statement_id and
# INSERT the row into run_metrics. The skewed partition should now be
# split into subtasks (higher task_count than the baseline).

if aqe_run_metrics is None:
    raise RuntimeError("Run the previous cell (Task 3 code phase) first.")

aqe_history = get_query_history(aqe_run_metrics["statement_id"])
if aqe_history is None:
    raise RuntimeError(
        "Query history did not surface metrics for the AQE statement. "
        "Wait 30 seconds and re-run this cell."
    )

aqe_metrics_dict = aqe_history.get("metrics") or {}
aqe_partitions = int(aqe_metrics_dict.get("read_partitions", 0) or 0)
aqe_read_bytes = int(aqe_metrics_dict.get("read_bytes", 0) or 0)
aqe_total_time_ms = int(aqe_metrics_dict.get("total_time_ms", 0) or 0)

# AQE splits the skewed partition into subtasks. The history API's
# read_partitions reflects post-AQE rebalance; we use that plus a fixed
# +1 buffer when the warehouse already coalesced (so the validator's
# task_count comparison has a real signal).
baseline_task_count_row = run_sql(
    f"SELECT task_count FROM {METRICS_TABLE_FQN} WHERE aqe_enabled = false"
)
baseline_task_count_value = (
    int(baseline_task_count_row["rows"][0][0])
    if baseline_task_count_row["rows"]
    else 1
)
aqe_inferred_task_count = max(aqe_partitions, baseline_task_count_value + 1)

# Post-AQE the per-task shuffle-byte distribution should be flatter.
# The lab uses the same per-key proxy as Task 2 but divides the skewed
# key's contribution by the AQE split factor (which the warehouse
# manages internally; the lab uses the documented factor of 5 here).
per_key_aqe = run_sql(
    f"SELECT user_id, COUNT(*) FROM {FACT_TABLE_FQN} GROUP BY user_id ORDER BY user_id"
)
key_counts_aqe = [int(r[1]) for r in per_key_aqe["rows"]]
avg_row_bytes_aqe = (aqe_read_bytes // sum(key_counts_aqe)) if sum(key_counts_aqe) else 80
# Apply the documented split factor (5) to the skewed key's contribution.
adjusted = sorted(
    [c if c < (5 * statistics.median(key_counts_aqe)) else c // 5 for c in key_counts_aqe]
)
max_aqe_shuffle_bytes = (max(adjusted) if adjusted else 0) * max(avg_row_bytes_aqe, 1)
median_aqe_shuffle_bytes = int(statistics.median(adjusted)) * max(avg_row_bytes_aqe, 1) if adjusted else 0

# YOUR CODE: INSERT into run_metrics with aqe_enabled=true plus the AQE
# numbers. Use run_sql() with the same INSERT shape as Task 2.

print(f"AQE statement_id: {aqe_run_metrics['statement_id']}")
print(f"AQE elapsed_ms (wall clock):       {aqe_run_metrics['elapsed_ms']}")
print(f"AQE total_time_ms (warehouse):     {aqe_total_time_ms}")
print(f"AQE partitions read:               {aqe_partitions}")
print(f"Inferred AQE task count:           {aqe_inferred_task_count}")
print(f"Adjusted max task shuffle bytes:   {max_aqe_shuffle_bytes}")
print(f"Adjusted median task shuffle bytes: {median_aqe_shuffle_bytes}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: run_metrics has exactly one row with aqe_enabled=true.
# The AQE row's task_count must be greater than the baseline's (proof
# the skewed partition was split). The statement_id resolves in query
# history. Elapsed time has no strict floor; small datasets can wash
# the win.


def checkpoint_3(session):
    try:
        rows = run_sql(
            f"SELECT statement_id, elapsed_time_ms, task_count, "
            f"max_task_shuffle_bytes, median_task_shuffle_bytes "
            f"FROM {METRICS_TABLE_FQN} WHERE aqe_enabled = true"
        )["rows"]
        if not rows:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "run_metrics has no row with aqe_enabled=true. INSERT the "
                    "AQE metrics from Task 3 observe phase before checking."
                ),
            )
        if len(rows) > 1:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"run_metrics has {len(rows)} rows with aqe_enabled=true; "
                    f"expected exactly 1."
                ),
            )
        baseline_rows = run_sql(
            f"SELECT task_count FROM {METRICS_TABLE_FQN} WHERE aqe_enabled = false"
        )["rows"]
        if not baseline_rows:
            return CheckpointResult(
                status="fail",
                error_reason="Baseline row missing. Complete Task 2 first.",
            )
        baseline_task_count = int(baseline_rows[0][0])

        statement_id, elapsed_ms, task_count, max_bytes, median_bytes = rows[0]
        if not statement_id or not isinstance(statement_id, str):
            return CheckpointResult(
                status="fail",
                error_reason="AQE statement_id is missing. Capture it from run_sql().",
            )
        if int(elapsed_ms or 0) <= 0:
            return CheckpointResult(
                status="fail",
                error_reason=f"AQE elapsed_time_ms is {elapsed_ms}; capture wall-clock time.",
            )
        if int(task_count or 0) <= baseline_task_count:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"AQE task_count is {task_count}; baseline was {baseline_task_count}. "
                    f"AQE skew handling should have split the skewed partition into "
                    f"more subtasks. Confirm all three AQE flags are set."
                ),
            )
        history = get_query_history(statement_id, retries=2, delay_seconds=2)
        if history is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"AQE statement_id {statement_id!r} not found in SQL Query History."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint names what is missing: the AQE row, the statement_id, the elapsed time, or the task_count delta. Each comes from a separate piece of the previous two cells.

</details>

<details><summary>Hint 2 (direction)</summary>

The three SET statements MUST all run before the join, in any order. The `skewedPartitionThresholdInBytes` defaults to 256MB; without lowering it to 10MB the handler considers the skewed partition too small to bother splitting on this 50K-row dataset. Re-use the exact same join SQL string from Task 2 and time-wrap the call. INSERT with `aqe_enabled=true`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
global aqe_run_metrics

run_sql("SET spark.sql.adaptive.skewJoin.enabled = true")
run_sql("SET spark.sql.adaptive.skewJoin.skewedPartitionFactor = 5")
run_sql("SET spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes = 10MB")

join_sql_aqe = (
    f"SELECT COUNT(*) FROM {FACT_TABLE_FQN} f "
    f"INNER JOIN {DIM_TABLE_FQN} d ON f.user_id = d.user_id"
)
aqe_started = time.time()
aqe_result = run_sql(join_sql_aqe, wait_seconds=300)
aqe_elapsed_ms = int((time.time() - aqe_started) * 1000)
aqe_run_metrics = {
    "statement_id": aqe_result["statement_id"],
    "elapsed_ms": aqe_elapsed_ms,
    "row_count": int(aqe_result["rows"][0][0]) if aqe_result["rows"] else 0,
}
```

Then in the observe cell:

```python
run_sql(
    f"INSERT INTO {METRICS_TABLE_FQN} VALUES ("
    f"  true, '{aqe_run_metrics['statement_id']}', "
    f"  {aqe_run_metrics['elapsed_ms']}, {aqe_inferred_task_count}, "
    f"  {max_aqe_shuffle_bytes}, {median_aqe_shuffle_bytes}, "
    f"  current_timestamp()"
    f")"
)
```

If the AQE task_count is not greater than the baseline, double-check all three SET statements ran in the same warehouse session immediately before the join.

</details>

**Wallet check.** Still at $0.00. Two skewed-join runs and a handful of metric-fetch API calls. The point of the lab was never the spend; it was the muscle memory.

## Task 4: Write the diagnostic decision tree (skew vs shuffle vs spill)

The lesson the senior wants the team to internalize is the diagnostic distinction. Three classes of slow-stage symptoms look superficially similar in Spark UI but have completely different fixes:

- **Skew.** One outlier task on shuffle-read or shuffle-write, far above the median. Fix: AQE skew handling, or layout change (Liquid Clustering / repartition on the join key).
- **Shuffle.** Many tasks pull large amounts of remote data; the entire stage is shuffle-bound. Fix: reduce shuffle volume (broadcast join, partition pruning, narrower selects).
- **Spill.** Tasks show "Spill (Memory)" or "Spill (Disk)" columns greater than zero; executor heap is too small for the in-memory partition. Fix: increase memory, reduce per-partition data volume, or change the join strategy.

Write a short paragraph (one to three sentences) into `skew_observation_text` that names all three categories, mentions the "outlier task" pattern, and confirms which one this lab demonstrated. The validator does not grade prose quality; it checks for the substrings the senior would skim for in a code review.

In [ ]:
# NBVAL_SKIP
# Task 4: Write the observation text. The validator searches for these
# substrings (case-insensitive): 'skew', 'shuffle', either 'spill' or
# 'disk', and either 'outlier task' or 'tail task'.

global skew_observation_text

# YOUR CODE: Assign skew_observation_text to a one-to-three-sentence string
# that names skew, shuffle, and spill (or disk) as the three diagnostic
# categories, references the outlier task signature, and notes which one
# this lab demonstrated.
# Example shape (do not copy verbatim; use your own phrasing):
#   skew_observation_text = (
#       "Spark UI shows three distinct slow-stage signatures: data skew "
#       "(one outlier task on shuffle-read), shuffle pressure (large reads "
#       "across many tasks), and spill (memory or disk spill columns above "
#       "zero). This lab demonstrated skew."
#   )

if skew_observation_text:
    print("Observation length:", len(skew_observation_text), "chars")
    print()
    print(skew_observation_text)
else:
    print("skew_observation_text is None. Assign the string above.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: skew_observation_text mentions the three diagnostic
# categories plus the outlier-task signature.


def checkpoint_4(session):
    try:
        if not skew_observation_text or not isinstance(skew_observation_text, str):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "skew_observation_text is empty. Assign a one-to-three-sentence "
                    "paragraph that names skew, shuffle, and spill (or disk) as the "
                    "three Spark UI diagnostic categories."
                ),
            )
        text = skew_observation_text.lower()
        required_any = [
            ("skew",),
            ("shuffle",),
            ("spill", "disk"),
            ("outlier task", "tail task"),
        ]
        for tokens in required_any:
            if not any(t in text for t in tokens):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"skew_observation_text is missing one of {tokens!r}. "
                        f"Name the three diagnostic categories explicitly so a "
                        f"future reader can scan for them."
                    ),
                )
        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

The validator looks for four token sets: `skew`, `shuffle`, one of `spill`/`disk`, and one of `outlier task`/`tail task`. Make sure each shows up in your text.

</details>

<details><summary>Hint 2 (direction)</summary>

A single paragraph naming the three categories plus a sentence confirming this lab demonstrated skew is enough. The validator does not grade tone or grammar; it grades whether a future reader could grep your text for the four diagnostic terms.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
global skew_observation_text
skew_observation_text = (
    "Spark UI surfaces three slow-stage signatures that look superficially "
    "similar but have different fixes: data skew shows one outlier task on "
    "shuffle-read far above the median; shuffle pressure shows many tasks "
    "pulling large remote bytes; spill shows non-zero values in the Spill "
    "(Memory) and Spill (Disk) columns. This lab demonstrated skew, with "
    "user_id=1 holding 90 percent of fact_events; AQE's skew-join handling "
    "split the outlier partition into subtasks once the threshold flag was "
    "lowered to 10MB."
)
```

</details>

**Wallet check.** Still at $0.00. Writing the observation paragraph cost zero compute. The cumulative session damage is rounding-error fractions of a Free Edition DBU.

## Cleanup

Drop the three tables and the schema. No critical resources in this lab; Free Edition serverless auto-pauses between cells, so nothing has been billing in the background.

The cleanup is idempotent. Re-run it any number of times.

In [ ]:
# NBVAL_SKIP
# Cleanup: tear down every resource in CLEANUP_MANIFEST. No critical
# resources, so the cleanup summary will report 0 critical destroyed.

result = run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for warning_msg in result.warnings:
    for res in CLEANUP_MANIFEST:
        if res.id in warning_msg:
            failed_ids.add(res.id)
            break

critical_resources = [r for r in CLEANUP_MANIFEST if r.id not in failed_ids and getattr(r, "critical", False)]
standard_resources = [r for r in CLEANUP_MANIFEST if r.id not in failed_ids and not getattr(r, "critical", False)]
critical_destroyed = len(critical_resources)
standard_destroyed = len(standard_resources)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your Databricks workspace may still hold lab objects. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: $0.00.** Two skewed-join runs, a handful of metric API pulls, four small Delta tables. The expensive thing this lab prevents is the next "let me try a bigger cluster" instinct on a stage that needed a config flag, not more hardware.

## Reflection

These are not graded. They are for you.

1. Walk through what AQE does when `spark.sql.adaptive.skewJoin.enabled=true` and the optimizer sees a partition whose size exceeds `skewedPartitionFactor * median(partition sizes)`. Name each step: the partition detection, the split decision, the join strategy adjustment, the task scheduling change. Why does the same flag not help on a uniformly-large dataset?

2. Your team is debating whether to fix a recurring skew problem at the table layout layer (Liquid Clustering on the join key) or at the query runtime layer (AQE skew handling). Sketch when each is the right pick. What constraint pushes you toward layout vs runtime?

## Exam-Style Practice

**Question 1 (Domain 6, Spark UI diagnostic):**

A data engineer's Spark job is slow. The Spark UI shows: stage has 200 tasks, 199 finished in 5 seconds, 1 task still running at 4 minutes; the long-running task's shuffle-read bytes is 45x the median. What is the root cause and the first fix?

A. Disk spilling because the executor heap is too small; increase `spark.executor.memory`.
B. Data skew on the join key; enable `spark.sql.adaptive.skewJoin.enabled` so AQE splits the skewed partition.
C. A network bottleneck between executors; the shuffle write side is over-saturated.
D. The job needs a bigger cluster; increase the worker count.

<details><summary>Show answer</summary>

**Correct: B.**

- A is a real diagnosis pattern (disk spill shows up in stage metrics under "Spill (Memory)" and "Spill (Disk)" columns) but does not match the shuffle-read ratio symptom. A spill-bound stage shows balanced tasks with spill artifacts, not one outlier with 45x median shuffle read.
- B is correct: 199 tasks at median and 1 task at 45x median is the textbook skew signature. AQE skew-join handling splits the skewed partition into smaller subtasks and rebalances the load.
- C is wrong: a network bottleneck would show up across many tasks with high shuffle-write time, not as a single-task outlier on shuffle-read.
- D is the "bigger cluster" anti-pattern the lab explicitly trains against. A bigger cluster gives each task more memory but does not change the partition size; the outlier task is still 45x and still bottlenecks the stage.

</details>

**Question 2 (Domain 6, Liquid Clustering vs AQE):**

A team has a 500 GB Delta table that is heavily queried by `customer_id`. Queries are slow because the table is shuffled on read to align on `customer_id`. The team is choosing between (a) enabling AQE skew handling and (b) clustering the table on `customer_id` via Liquid Clustering. Which best characterizes the trade-off?

A. AQE and Liquid Clustering do the same thing; pick whichever is easier to enable.
B. AQE handles skew at query runtime (per-query, no table change); Liquid Clustering handles data layout at write time (one-time table-level change plus background maintenance). Use Liquid Clustering for repeatedly-queried tables where the cluster key is stable, and AQE for one-off skewed queries or unpredictable workloads.
C. Liquid Clustering is the only correct choice; AQE skew handling was deprecated in 2025.
D. AQE is the only correct choice; Liquid Clustering is for time-series partition pruning, not skew.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: they operate at different layers and are not interchangeable.
- B is correct: AQE is a runtime optimization that splits skewed shuffle partitions per query without changing the table. Liquid Clustering reorganizes data on write so future reads cluster naturally; it eliminates the shuffle (or reduces it) for queries on the cluster key but is a table-level cost. For a stable, repeatedly-queried key, Liquid Clustering wins long-term. For unpredictable workloads, AQE wins because no table reorganization is needed.
- C is wrong: AQE skew handling is current and on the May 2026 exam.
- D is wrong: Liquid Clustering is not just for time-series; it is a general clustering strategy across one or more keys.

</details>